In [1]:
import pandas as pd
import numpy as np
import torch
import joblib
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import os

W0503 23:36:01.618000 6824 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
path_gold = '../clustering_tests/test_samples/GOLD_STANDARD_500_done.csv'

path_raw = '../../data/messages.csv'

In [3]:
df_gold = pd.read_csv(path_gold)
df_gold = df_gold.rename(columns={'manual_label': 'label'})

print(f"Загружен Gold Standard: {len(df_gold)} строк")

Загружен Gold Standard: 500 строк


In [4]:
df_raw_full = pd.read_csv(path_raw)
#исключаем ID, которые есть в золотом стандарте
gold_ids = df_gold['id'].unique()
df_other_pool = df_raw_full[~df_raw_full['id'].isin(gold_ids)]
print(f"Доступно строк для добавления 'мусора': {len(df_other_pool)}")

Доступно строк для добавления 'мусора': 45342


In [5]:
df_other = df_other_pool.sample(n=60, random_state=42).copy()
df_other['label'] = 'OTHER'

df_final = pd.concat([
    df_gold[['text', 'label']], 
    df_other[['text', 'label']]
], ignore_index=True)

# очистка от пустых значений
df_final = df_final.dropna(subset=['text', 'label'])

print(f"Итоговый датасет для Модели А: {len(df_final)} строк")
print(f"Распределение классов:\n{df_final['label'].value_counts()}")

Итоговый датасет для Модели А: 560 строк
Распределение классов:
label
OTHER                                    60
INFRA_CPU_HIGH_USAGE                     42
INFRA_ORACLE_TNS_LISTENER_ERROR          42
APP_API_AUTH_500_ERROR                   38
INFRA_TLS_HANDSHAKE_OR_PORT_443          35
INFRA_POSTGRES_WALARCHIVE_FULL           34
INFRA_POSTGRES_REPLICATION_LAG           33
INFRA_POSTGRES_SERVICE_STATUS_FAILED     32
INFRA_NETWORK_OR_REMOTE_TIMEOUT          29
INFRA_HYPERVISOR_HEARTBEAT_LOST          27
INFRA_K8S_POD_RESTART_FAILURE            27
APP_CARD_AUTH_BUSINESS_ERROR             25
INFRA_HARDWARE_OR_LOG_THRESHOLD          21
INFRA_NETWORK_LATENCY_RTT                21
INFRA_SYSTEM_WIDESPREAD_FAILURE_ALERT    20
INFRA_DISK_SPACE_CRITICAL                12
APP_INTERNAL_ERROR_CODES                 10
INFRA_MESSAGE_BROKER_QUEUE_OVERFLOW       9
INFRA_STORAGE_RO_OR_MOUNT_FAIL            9
APP_FINBANK_FUNCTIONAL_UNAVAILABILITY     8
INFRA_MONITORING_AGENT_DOWN               7
INFRA_

In [7]:
'''
Грязные метки: INFRA_ORACLE_TNS_LISTENER_ERROR\r\n с 1 примером -это дубль основной метки.
Классы-одиночки (N=1): StratifiedKFold и train_test_split с параметром stratify выдадут ошибку, если в классе всего 1 пример.
Перекос в OTHER: 60 примеров OTHER при 1-2 примерах в других классах.
'''

'\nГрязные метки: INFRA_ORACLE_TNS_LISTENER_ERROR\r\n с 1 примером -это дубль основной метки.\nКлассы-одиночки (N=1): StratifiedKFold и train_test_split с параметром stratify выдадут ошибку, если в классе всего 1 пример.\nПерекос в OTHER: 60 примеров OTHER при 1-2 примерах в других классах.\n'

In [8]:
# уборка пробелов и переносов строк (\r\n)
df_final['label'] = df_final['label'].str.strip()

# Чтобы StratifiedSplit (на 100 тест и 5 фолдов) не упал, 
# в каждом классе должно быть минимум 6 примеров (1 в тест + 5 для фолдов)
class_counts = df_final['label'].value_counts()
rare_classes = class_counts[class_counts < 6].index

In [9]:
print(f"Классы, которые мы будем дублировать (oversampling): {list(rare_classes)}")

for cls in rare_classes:
    curr_count = class_counts[cls]
    needed = 6 - curr_count
    # дублируем существующие примеры класса
    extra_rows = df_final[df_final['label'] == cls].sample(n=needed, replace=True, random_state=42)
    df_final = pd.concat([df_final, extra_rows], ignore_index=True)

print(f"Новое распределение:\n{df_final['label'].value_counts()}")

Классы, которые мы будем дублировать (oversampling): ['INFRA_NETWORK_BANDWIDTH_EXCEEDED', 'TEST', 'APP_ZOD', 'EMAIL', 'INFRA_WEB_SERVER_IIS_OWA_FAIL', 'INFRA_JDBC_CONNECTION_POOL_EXHAUSTED', 'INFRA_MONITORING_METRIC_MISSING']
Новое распределение:
label
OTHER                                    60
INFRA_ORACLE_TNS_LISTENER_ERROR          43
INFRA_CPU_HIGH_USAGE                     42
APP_API_AUTH_500_ERROR                   38
INFRA_TLS_HANDSHAKE_OR_PORT_443          35
INFRA_POSTGRES_WALARCHIVE_FULL           34
INFRA_POSTGRES_REPLICATION_LAG           33
INFRA_POSTGRES_SERVICE_STATUS_FAILED     32
INFRA_NETWORK_OR_REMOTE_TIMEOUT          29
INFRA_HYPERVISOR_HEARTBEAT_LOST          27
INFRA_K8S_POD_RESTART_FAILURE            27
APP_CARD_AUTH_BUSINESS_ERROR             25
INFRA_HARDWARE_OR_LOG_THRESHOLD          21
INFRA_NETWORK_LATENCY_RTT                21
INFRA_SYSTEM_WIDESPREAD_FAILURE_ALERT    20
INFRA_DISK_SPACE_CRITICAL                12
APP_INTERNAL_ERROR_CODES                 10

In [10]:
print(f"Изменённое количество золотого стандарта:\n{len(df_final)}")

Изменённое количество золотого стандарта:
590


In [11]:
'''
Так как теперь 590 строк и в каждом классе минимум 6 примеров, всё разделится идеально:
В Hold-out Test (100 строк) улетит по 1 примеру каждого редкого класса.
В Train/Val останется по 5 примеров каждого редкого класса.
При 5-Fold CV эти 5 примеров распределятся так: 4 в обучение, 1 в валидацию внутри каждого фолда.
'''

'\nТак как теперь 590 строк и в каждом классе минимум 6 примеров, всё разделится идеально:\nВ Hold-out Test (100 строк) улетит по 1 примеру каждого редкого класса.\nВ Train/Val останется по 5 примеров каждого редкого класса.\nПри 5-Fold CV эти 5 примеров распределятся так: 4 в обучение, 1 в валидацию внутри каждого фолда.\n'

In [13]:
MODEL_NAME = "cointegrated/rubert-tiny"
SEED = 42
K_FOLDS = 5

all_labels = sorted(df_final['label'].unique())
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}
df_final['label_id'] = df_final['label'].map(label2id)

#энкодер для api
joblib.dump(label2id, "label_encoder_model_a.joblib")


['label_encoder_model_a.joblib']

In [17]:
#  Hold-out
train_val_df, test_df = train_test_split(
    df_final, 
    test_size=100, 
    stratify=df_final['label_id'], 
    random_state=SEED
)

# сброс индексов
train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Размер train_val (для кросс-валидации): {len(train_val_df)}")
print(f"Размер test (hold-out): {len(test_df)}")

# Проверка: попал ли OTHER в тест
other_in_test = len(test_df[test_df['label'] == 'OTHER'])
print(f"Примеров OTHER в тесте: {other_in_test}")

Размер train_val (для кросс-валидации): 490
Размер test (hold-out): 100
Примеров OTHER в тесте: 10


In [18]:
# токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# создание модели со сбросом весов перед каждым фолдом
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=len(all_labels),
        id2label=id2label,
        label2id=label2id
    )

In [19]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
cv_metrics = []

# подготовка данных для сплиттера
X = train_val_df['text'].values
y = train_val_df['label_id'].values

print(f"Запуск {K_FOLDS}-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- FOLD {fold+1}/{K_FOLDS} ---")
    
    # датасеты для фолда
    train_ds = Dataset.from_pandas(train_val_df.iloc[train_idx][['text', 'label_id']].rename(columns={'label_id':'label'}))
    val_ds = Dataset.from_pandas(train_val_df.iloc[val_idx][['text', 'label_id']].rename(columns={'label_id':'label'}))
    
    train_ds = train_ds.map(tokenize_func, batched=True)
    val_ds = val_ds.map(tokenize_func, batched=True)

    # настройки обучения для фолда
    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=10,
        report_to="none"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {"f1_macro": f1_score(labels, predictions, average="macro")}

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    # считаем финальную метрику фолда
    res = trainer.evaluate()
    cv_metrics.append(res['eval_f1_macro'])
    print(f"Fold {fold+1} F1-Macro: {res['eval_f1_macro']:.4f}")

print(f"\nКросс-валидация завершена!")
print(f"Средний F1-Macro: {np.mean(cv_metrics):.4f} (+/- {np.std(cv_metrics):.4f})")

Запуск 5-Fold Cross-Validation...

--- FOLD 1/5 ---


Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

D:\projects\context\contextual_search\venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but n

Epoch,Training Loss,Validation Loss,F1 Macro
1,3.283200,3.220953,0.008434
2,3.150300,3.121270,0.015189
3,3.071700,3.045372,0.056524
4,2.975500,2.993419,0.093772
5,2.959800,2.973487,0.093772


...

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

D:\projects\context\contextual_search\venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but n

Epoch,Training Loss,Validation Loss,F1 Macro
1,3.273700,3.224302,0.014916
2,3.129600,3.125874,0.006506
3,3.075800,3.048598,0.118988
4,2.973000,2.992561,0.104117
5,2.960800,2.973511,0.137309


D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
D:\projects\context\contextu

Fold 5 F1-Macro: 0.1373

Кросс-валидация завершена!
Средний F1-Macro: 0.1209 (+/- 0.0169)


In [21]:
print("\nПодготовка финальных датасетов...")

# Dataset из train_val_df (все 490 строк)
full_train_ds = Dataset.from_pandas(train_val_df[['text', 'label_id']].rename(columns={'label_id':'label'}))
full_train_ds = full_train_ds.map(tokenize_func, batched=True)

# Dataset из test_df (100 строк Hold-out)
test_ds = Dataset.from_pandas(test_df[['text', 'label_id']].rename(columns={'label_id':'label'}))
test_ds = test_ds.map(tokenize_func, batched=True)


print("\nОбучаем финальную Модель А (усиленный режим)...")

# увеличение количества эпох и правка темпа обучения
final_args = TrainingArguments(
    output_dir="./saved_model_A_final",
    num_train_epochs=30,           # поднятие эпох до 30
    per_device_train_batch_size=8,  # уменьшение батча, чтобы обновлений было больше
    learning_rate=1e-4,            # чуть агрессивнее обучение
    weight_decay=0.01,
    lr_scheduler_type="cosine",    # плавное затухание скорости
    warmup_ratio=0.1,
    report_to="none",
    save_strategy="no",
    logging_steps=20
)

final_trainer = Trainer(
    model_init=model_init,
    args=final_args,
    train_dataset=full_train_ds,
    # нет использования eval_dataset здесь, так как учим на всем доступном
)

final_trainer.train()

final_trainer.save_model("./saved_model_A_final")
tokenizer.save_pretrained("./saved_model_A_final")

preds = final_trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = test_df['label_id'].values

print("\nИТОГОВЫЙ ОТЧЕТ МОДЕЛИ А (Hold-out):")
print(classification_report(y_true, y_pred, target_names=all_labels))


Подготовка финальных датасетов...


Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


Обучаем финальную Модель А (усиленный режим)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,3.365500
40,3.341200
60,3.307300
80,3.231000
100,3.184000
120,3.090200
140,2.965100
160,2.802600
180,2.713900
200,2.394500



ИТОГОВЫЙ ОТЧЕТ МОДЕЛИ А (Hold-out):
                                       precision    recall  f1-score   support

               APP_API_AUTH_500_ERROR       0.75      1.00      0.86         6
         APP_CARD_AUTH_BUSINESS_ERROR       1.00      1.00      1.00         4
APP_FINBANK_FUNCTIONAL_UNAVAILABILITY       1.00      1.00      1.00         1
             APP_INTERNAL_ERROR_CODES       0.00      0.00      0.00         2
                              APP_ZOD       1.00      1.00      1.00         1
                                EMAIL       1.00      1.00      1.00         1
             INFRA_CONNECTION_REFUSED       0.00      0.00      0.00         1
                 INFRA_CPU_HIGH_USAGE       0.78      1.00      0.88         7
            INFRA_DISK_SPACE_CRITICAL       0.50      1.00      0.67         2
      INFRA_HARDWARE_OR_LOG_THRESHOLD       1.00      0.75      0.86         4
      INFRA_HYPERVISOR_HEARTBEAT_LOST       0.83      1.00      0.91         5
 INFRA_JDBC_CO

D:\projects\context\contextual_search\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\projects\context\contextual_search\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\projects\context\contextual_search\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

In [22]:
DATA_EXPORT_PATH = "../data/processed"
os.makedirs(DATA_EXPORT_PATH, exist_ok=True)

# сохранение Hold-out с оригинальными метками и текстами
test_df.to_csv(os.path.join(DATA_EXPORT_PATH, "gold_test_holdout.csv"), index=False)
print(f"Тестовый набор (100 строк) сохранен в: gold_test_holdout.csv")

Тестовый набор (100 строк) сохранен в: gold_test_holdout.csv


In [25]:
used_ids_gold = df_gold['id'].unique().tolist()
used_ids_other = df_other['id'].unique().tolist()
all_used_ids = set(used_ids_gold + used_ids_other)

print(f"Всего исключаем ID: {len(all_used_ids)}")

path_pseudo = '../clustering_tests/test_samples/TRAIN_DATASET_WITH_LABELS.csv' 
df_pseudo = pd.read_csv(path_pseudo)

print(f"Колонки в файле: {df_pseudo.columns.tolist()}")

possible_label_cols = [c for c in df_pseudo.columns if c not in ['id', 'text', 'processed_text']]
source_label_col = possible_label_cols[0] 
print(f"Используем колонку '{source_label_col}' как источник псевдоразметки")

df_weak_train = df_pseudo[~df_pseudo['id'].isin(all_used_ids)].copy()

allowed_labels = set(all_labels) # all_labels берем из предыдущих ячеек

def sync_labels(val):
    val = str(val).strip()
    if val in allowed_labels:
        return val
    else:
        return 'OTHER'

df_weak_train['label'] = df_weak_train[source_label_col].apply(sync_labels)

print(f"Итого строк для обучения Модели Б: {len(df_weak_train)}")
print(f"Классов в Модели Б: {df_weak_train['label'].nunique()}")

export_file_path = os.path.join(DATA_EXPORT_PATH, "weak_train_45k.csv")
df_weak_train[['text', 'label']].to_csv(export_file_path, index=False)

print(f"Данные успешно экспортированы в: {export_file_path}")

Всего исключаем ID: 560
Колонки в файле: ['id', 'text', 'parsed_text', 'clean_text', 'cluster_label', 'pseudo_label']
Используем колонку 'parsed_text' как источник псевдоразметки
Итого строк для обучения Модели Б: 44294
Классов в Модели Б: 1
Данные успешно экспортированы в: ../data/processed\weak_train_45k.csv
